<a href="https://colab.research.google.com/github/royalgarter/cofog-panel/blob/main/cofog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COFOG-Panel: Harmonizing IMF GFS COFOG Data into Reproducible Panel Datasets

This notebook automates the setup, installation, and execution of the COFOG-Panel pipeline

In [10]:
# @title 1. Setup Environment and Install Dependencies

!git clone https://github.com/royalgarter/cofog-panel.git
%cd cofog-panel

# Install dependencies from requirements.txt
!pip install -r requirements.txt

# Install the package in editable mode to use the `cofog-panel` CLI command
!pip install -e .

Cloning into 'cofog-panel'...
remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 240 (delta 11), reused 1 (delta 0), pack-reused 215 (from 2)
Receiving objects: 100% (240/240), 51.19 MiB | 24.89 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/cofog-panel/cofog-panel
Obtaining file:///content/cofog-panel/cofog-panel
  Preparing metadata (setup.py) ... done
  Attempting uninstall: cofog-panel
    Found existing installation: cofog-panel 0.1.0
    Uninstalling cofog-panel-0.1.0:
      Successfully uninstalled cofog-panel-0.1.0
  Running setup.py develop for cofog-panel


In [11]:
# @title 2. Prepare Data
# @markdown Ensure your input files are placed in the ./data/ directory:
# @markdown - gfs_raw_data.xlsx (Main COFOG source)
# @markdown - country_codes.xlsx (Country mapping file)


import os

# Create necessary directories
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('intermediate_splits', exist_ok=True)

print("Directories verified.")

Directories verified.


In [12]:
# @title 3. Input Form
# @markdown * **'COFOG' Dropdown**: This dropdown lists all the unique COFOG (Classification of the Functions of Government) values found in your gfs_raw_data.xlsx file. You can select a specific spending function from this list.
# @markdown * **'TYPE_OF_TRANSFORMATION' Dropdown**: This dropdown shows the unique data transformation types (e.g., 'Percent of total outlays', 'Percent of GDP') available in your data. You can choose how the data is represented.
# @markdown * **'SECTOR' Dropdown**: This dropdown contains all the unique sectors (e.g., 'Central government', 'Local Government') present in your Excel file. You can filter the data by a specific sector.
# @markdown * **'Data Column Name' Text Input**: This is an empty box where you can type a custom name for a new data column that will be created later in the processing pipeline.

import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# Define the path to the raw GFS data Excel file
file_path = './data/gfs_raw_data.xlsx'
# Load the data into a pandas DataFrame
df = pd.read_excel(file_path)

# Create Dropdown for COFOG values
cofog_values = df['COFOG'].unique()
dropdown_cofog = widgets.Dropdown(
    options=cofog_values,
    description='COFOG:',
)
display(dropdown_cofog)

# Create Dropdown for TYPE_OF_TRANSFORMATION values
transformation_values = df['TYPE_OF_TRANSFORMATION'].unique()
dropdown_transformartion = widgets.Dropdown(
    options=transformation_values,
    description='Transformation:',
)
display(dropdown_transformartion)

# Create Dropdown for SECTOR values
sector_values = df['SECTOR'].unique()
dropdown_sector = widgets.Dropdown(
    options=sector_values,
    description='Sector:',
)
display(dropdown_sector)

# Create Text Input for new data column name
input_data_col = widgets.Text(
    placeholder='Enter new data column name here',
    description='Column:'
)
display(input_data_col)

Dropdown(description='Choose COFOG:', options=('Fuel and energy', 'Tertiary education', 'Social protection', '…

Dropdown(description='Choose Transformation Type:', options=('Percent of total outlays', 'Percent of GDP', 'Do…

Dropdown(description='Choose Sector:', options=('Social security funds', 'State Government', 'Extrabudgetary c…

Text(value='', description='Data Column Name:', placeholder='Enter new data column name here')

In [14]:
print(dropdown_cofog.value)
print(dropdown_transformartion.value)
print(dropdown_sector.value)
print(input_data_col.value)

Defence
Percent of GDP
General government
DATA_DEFENCE


In [16]:
# @title 4. Orchestrator: Run Full Pipeline
# @markdown This command automates Stages 1 through 5 sequentially.

!cofog-panel run \
    --source-file ./data/gfs_raw_data.xlsx \
    --lookup-file ./data/country_codes.xlsx \
    --cofog "Defence" \
    --sector "General government" \
    --output-cols "DATA_DEFENCE"

--- Starting FULL LOCAL COFOG Pipeline Orchestration ---

[Stage 1/5] Checking COFOG Format...

COFOG FORMAT VERIFICATION REPORT (gfs_raw_data.xlsx)
🎉 SUCCESS: File header structure matches the required template.

[Stage 2/5] Checking Country Code Format...
✅ SUCCESS: Country lookup file structure is VALID.
   -> Total Rows: 249
   -> Columns: ['name', 'alpha-2', 'alpha-3', 'country-code', 'iso_3166-2', 'region', 'sub-region', 'intermediate-region', 'region-code', 'sub-region-code', 'intermediate-region-code']

[Stage 3/5] Seeding Master XLSX...

--- SEEDING MASTER: Processing country_codes.xlsx ---
✅ SUCCESS: Master XLSX file created successfully.
=> File Path: output/COFOG_MASTER_SCHEMA.xlsx
Master Path Established: output/COFOG_MASTER_SCHEMA.xlsx

[Stage 4/5] Splitting Data per Country (to Local XLSX)...

--- ETL & SPLIT: Processing gfs_raw_data.xlsx ---
Filtering data by Transformation: Percent of GDP and COFOGs: {'Defence'}...

--- Filtering complete. Total Processed: 372482. Tota

[link text](https://)### Individual Stages (Manual Execution)

In [15]:
# @title 5. Stage 1 & 2: Validation

!cofog-panel check-format --source-file ./data/gfs_raw_data.xlsx
!cofog-panel check-country-format --lookup-file ./data/country_codes.xlsx


COFOG FORMAT VERIFICATION REPORT (gfs_raw_data.xlsx)
🎉 SUCCESS: File header structure matches the required template.
✅ SUCCESS: Country lookup file structure is VALID.
   -> Total Rows: 249
   -> Columns: ['name', 'alpha-2', 'alpha-3', 'country-code', 'iso_3166-2', 'region', 'sub-region', 'intermediate-region', 'region-code', 'sub-region-code', 'intermediate-region-code']


In [17]:
# @title 5. Stage 3: Seeding the Master Schema
!cofog-panel seed-master --lookup-file ./data/country_codes.xlsx --output-dir ./output


--- SEEDING MASTER: Processing country_codes.xlsx ---
✅ SUCCESS: Master XLSX file created successfully.
=> File Path: output/COFOG_MASTER_SCHEMA.xlsx
::MASTER_PATH::output/COFOG_MASTER_SCHEMA.xlsx


In [18]:
# @title 5. Stage 4: ETL and Data Splitting
!cofog-panel split --source-file ./data/gfs_raw_data.xlsx --cofog "{dropdown_cofog.value}" --filter-type "{dropdown_transformartion.value}"


--- ETL & SPLIT: Processing gfs_raw_data.xlsx ---
Filtering data by Transformation: Percent of GDP and COFOGs: {'Defence'}...

--- Filtering complete. Total Processed: 372482. Total Valid: 1552 ---
Writing 8 rows for country: LUX...
-> Successfully wrote LUX.
Writing 8 rows for country: AUT...
-> Successfully wrote AUT.
Writing 8 rows for country: SWE...
-> Successfully wrote SWE.
Writing 8 rows for country: GRC...
-> Successfully wrote GRC.
Writing 8 rows for country: CYP...
-> Successfully wrote CYP.
Writing 8 rows for country: BGR...
-> Successfully wrote BGR.
Writing 8 rows for country: CHE...
-> Successfully wrote CHE.
Writing 8 rows for country: UKR...
-> Successfully wrote UKR.
Writing 8 rows for country: AGO...
-> Successfully wrote AGO.
Writing 8 rows for country: SOM...
-> Successfully wrote SOM.
Writing 8 rows for country: BIH...
-> Successfully wrote BIH.
Writing 8 rows for country: PRY...
-> Successfully wrote PRY.
Writing 8 rows for country: CPV...
-> Successfully wrote 

In [19]:
# @title 5. Stage 5: Harmonization and Aggregation
!cofog-panel aggregate --master-file ./output/COFOG_MASTER_SCHEMA.xlsx --folder-path ./intermediate_splits --data-col "{input_data_col.value}" --sector "{dropdown_sector.value}"


--- AGGREGATE STAGE: Processing files in intermediate_splits ---
✅ Master XLSX loaded: 8964 rows.
📂 Found 194 intermediate country XLSX files in intermediate_splits.
[1/194] Processing WSM -> OK (14 updates)
[2/194] Processing DZA -> OK (10 updates)
[3/194] Processing SYC -> OK (17 updates)
[4/194] Processing AZE -> OK (22 updates)
[5/194] Processing TGO -> OK (1 updates)
[6/194] Processing COL -> OK (8 updates)
[7/194] Processing TCD -> OK (0 updates)
[8/194] Processing KGZ -> OK (20 updates)
[9/194] Processing TJK -> OK (7 updates)
[10/194] Processing CIV -> OK (1 updates)
[11/194] Processing MEX -> OK (11 updates)
[12/194] Processing AUT -> OK (34 updates)
[13/194] Processing NAM -> OK (24 updates)
[14/194] Processing TON -> OK (1 updates)
[15/194] Processing GMB -> OK (1 updates)
[16/194] Processing HUN -> OK (34 updates)
[17/194] Processing GBR -> OK (34 updates)
[18/194] Processing IRL -> OK (34 updates)
[19/194] Processing URY -> OK (15 updates)
[20/194] Processing PAN -> OK (8